In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import re
import json
import math
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
import librosa

In [3]:
INPUT_DIR = "/kaggle/input/nppe-2-automatic-disfluency-restoration"
OUTPUT_DIR = "/kaggle/working"
AUDIO_DIR = os.path.join(INPUT_DIR, "downloaded_audios")
CACHE_DIR = os.path.join(OUTPUT_DIR, "asr_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

In [4]:
USE_AUDIO = True

# ASR_MODEL_NAME = "collabora/whisper-small-hindi"
ASR_MODEL_NAME = "collabora/whisper-medium-hindi"
SEQ2SEQ_MODEL = "ai4bharat/IndicBART"

SRC_LANG_TOKEN = "<2en>"
TGT_LANG_TOKEN = "<2hi>"

BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 5e-5
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512
GRAD_ACCUM_STEPS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [6]:
print("Loading CSVs...")
train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
print(f"Total Train samples: {len(train_df)} | Test samples: {len(test_df)}")

Loading CSVs...
Total Train samples: 900 | Test samples: 100


In [7]:
def load_disfluencies(path):
    df = pd.read_csv(path)
    return set(df["disfluency"].astype(str).tolist())

def remove_disfluencies(text, dis):
    if pd.isna(text):
        return ""
    words = text.split()
    return " ".join([w for w in words if w not in dis]).strip()

disfluencies = load_disfluencies(os.path.join(INPUT_DIR, "unique_disfluencies.csv"))

train_df["clean_transcript"] = train_df["transcript"].apply(
    lambda x: remove_disfluencies(x, disfluencies)
)
train_df = train_df.rename(columns={"transcript": "disfluent_transcript"})

In [8]:
train_df["id"] = train_df["id"].astype(str)
test_df["id"] = test_df["id"].astype(str)
train_df["path"] = train_df["id"].apply(lambda x: os.path.join(AUDIO_DIR, f"{x}.wav"))
test_df["path"] = test_df["id"].apply(lambda x: os.path.join(AUDIO_DIR, f"{x}.wav"))
train_df = train_df[["id", "clean_transcript", "disfluent_transcript", "path"]]
test_df = test_df.rename(columns={"transcript": "clean_transcript"})

In [9]:
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)
print(f"Training on {len(train_data)} samples, Validating on {len(val_data)} samples.")

Training on 810 samples, Validating on 90 samples.


In [10]:
ASR_CACHE_PATH = os.path.join(CACHE_DIR, "asr_cache.json")
if os.path.exists(ASR_CACHE_PATH):
    print("Loading existing ASR cache")
    with open(ASR_CACHE_PATH, "r") as f:
        asr_cache = json.load(f)
else:
    asr_cache = {}

In [ ]:
if USE_AUDIO:
    print(f"Loading Whisper ASR model: {ASR_MODEL_NAME}...")
    asr_processor = WhisperProcessor.from_pretrained(ASR_MODEL_NAME)
    asr_model = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_NAME).to(DEVICE)
    asr_model.eval()
    forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

    def audio_to_text(path):
        key = os.path.basename(path)
        if key in asr_cache:
            return asr_cache[key]
        if not os.path.exists(path):
            asr_cache[key] = ""
            return ""
        try:
            audio, sr = librosa.load(path, sr=16000)
            inputs = asr_processor(audio, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(DEVICE)
            with torch.no_grad():
                pred_ids = asr_model.generate(
                    input_features,
                    forced_decoder_ids=forced_decoder_ids,
                    max_length=256
                )
            text = asr_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].lower().strip()
        except Exception as e:
            print(f"Error processing {path}: {e}")
            text = ""
        asr_cache[key] = text
        return text

    if os.path.exists(ASR_CACHE_PATH):
        os.remove(ASR_CACHE_PATH)
        asr_cache = {}


    print("Generating ASR transcripts (train)")
    for p in tqdm(train_data["path"]):
        audio_to_text(p)
    print("GeneratingASR transcripts (validation)")
    for p in tqdm(val_data["path"]):
        audio_to_text(p)
    print("Generating ASR transcripts (test)")
    for p in tqdm(test_df["path"]):
        audio_to_text(p)
    print("Saving ASR cache")
    with open(ASR_CACHE_PATH, "w") as f:
        json.dump(asr_cache, f, ensure_ascii=False, indent=4)


In [12]:
def build_multimodal(clean, audio_file):
    clean = clean.strip() if isinstance(clean, str) else ""
    asr_text = asr_cache.get(os.path.basename(audio_file), "") if USE_AUDIO else ""
    if asr_text:
        return f"{clean} [ASR] {asr_text} </s> {SRC_LANG_TOKEN}"
    return f"{clean} </s> {SRC_LANG_TOKEN}"

In [ ]:
print(f"Loading Seq2Seq model: {SEQ2SEQ_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    SEQ2SEQ_MODEL,
    do_lower_case=False,
    use_fast=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(SEQ2SEQ_MODEL).to(DEVICE)

new_tokens = ["[ASR]", SRC_LANG_TOKEN, TGT_LANG_TOKEN]
tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
model.resize_token_embeddings(len(tokenizer))
    
HINDI_TOKEN_ID = tokenizer.convert_tokens_to_ids(TGT_LANG_TOKEN)
print(f"Hindi Token ID ('{TGT_LANG_TOKEN}') set to: {HINDI_TOKEN_ID}")

In [14]:
class MultiModalDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        multimodal_input = build_multimodal(row["clean_transcript"], row["path"])

        enc = tokenizer(
            multimodal_input,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
            return_attention_mask=True,
            add_special_tokens=False
        )

        item = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "id": row["id"]
        }

        if self.is_train:
            target_text = f"{TGT_LANG_TOKEN} {row['disfluent_transcript']} </s>"
            
            target = tokenizer(
                target_text,
                truncation=True,
                max_length=MAX_TARGET_LENGTH,
                add_special_tokens=False
            )
            item["labels"] = target["input_ids"]

        return item


def collate(batch):
    input_ids = [b["input_ids"] for b in batch]
    attn = [b["attention_mask"] for b in batch]

    padded = tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attn},
        padding="longest",
        return_tensors="pt"
    )

    out = {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
        "id": [b["id"] for b in batch]
    }

    if "labels" in batch[0]:
        labels = [b["labels"] for b in batch]
        lab_pad = tokenizer.pad({"input_ids": labels}, padding="longest", return_tensors="pt")["input_ids"]
        # -100 is the ignore_index for labels
        lab_pad[lab_pad == tokenizer.pad_token_id] = -100
        out["labels"] = lab_pad

    return out

In [15]:
train_ds = MultiModalDataset(train_data, True)
val_ds = MultiModalDataset(val_data, True)
test_ds = MultiModalDataset(test_df, False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [17]:
def train_epoch(epoch):
    model.train()
    total_loss = 0
    
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
        outputs = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            labels=batch["labels"].to(DEVICE)
        )
        loss = outputs.loss
        loss = loss / GRAD_ACCUM_STEPS 
        loss.backward()
        
        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if (i + 1) % GRAD_ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
    avg_train_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.6f}")

In [18]:
def evaluate(val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                labels=batch["labels"].to(DEVICE)
            )
            loss = outputs.loss
            total_loss += loss.item()

    avg_val_loss = total_loss / len(val_loader.dataset)
    return avg_val_loss

In [ ]:
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    train_epoch(epoch)
    val_loss = evaluate(val_loader)
    print(f"Epoch {epoch+1} Validation Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved to {MODEL_SAVE_PATH} (Val Loss: {val_loss:.6f})")
    else:
        print("Validation loss did not improve.")

In [ ]:
print("\nStarting inference")
if os.path.exists(MODEL_SAVE_PATH):
    print(f"Loading best model from {MODEL_SAVE_PATH}")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
else:
    print("Warning: No best model found. Using last epoch model.")
     
model.eval()
pred_ids = []
pred_texts = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        gen = model.generate(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            forced_bos_token_id=HINDI_TOKEN_ID,
            max_length=MAX_TARGET_LENGTH,
            num_beams=8,
            early_stopping=True
        )
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        pred_texts.extend(decoded)
        pred_ids.extend(batch["id"])

submission_raw = pd.DataFrame({"id": pred_ids, "transcript": pred_texts})
submission_raw["id"] = submission_raw["id"].astype(str)

final_sub = test_df[["id"]].merge(submission_raw, on="id", how="left")
final_sub["transcript"] = final_sub["transcript"].fillna("")

sub_path = os.path.join(OUTPUT_DIR, "submission.csv")
final_sub.to_csv(sub_path, index=False)

print("\nSaved submission:", sub_path)
print(final_sub.head())


Starting inference
Loading best model from /kaggle/working/best_model.pth


Predicting:   0%|          | 0/13 [00:00<?, ?it/s]You're using a AlbertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a AlbertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Predicting: 100%|██████████| 13/13 [00:17<00:00,  1.38s/it]


Saved submission: /kaggle/working/submission.csv
           id                                         transcript
0  8894265003     जैसे वो दरी वगेरा बना सकते हैं जैसे घर में जो 
1  8951729741                                              क्या 
2  4268956831  हम आप अपने हूं जो खास दोस्त रहता है उससे लड़ाई...
3  1819728609                     सही बात है जिसमें कुछ मोरल हो 
4  5456358058  तनकियों अं तनकियों के बारे में बात करना काम और...
